# Notebook 1 (Milvus): RAG Context Preparation
This notebook isolates the heavy 8B embedding model to pre-compute and retrieve RAG contexts using the local **Milvus Lite** vector database. This ensures zero VRAM overlap with the LLM generation phase.


## 1. Install Dependencies


In [ ]:
%%capture
!pip install sentence-transformers pymilvus datasets scikit-learn


## 2. Load Dataset & Sample


In [ ]:
import json
import os
import random
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient

test_file_path = "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/test-data-reformat.json"

test_data = []
if os.path.exists(test_file_path):
    with open(test_file_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
else:
    print(f"File not found: {test_file_path}")

# Extract unique context fields
unique_contexts = list(set([item.get('context', '').strip() for item in test_data if item.get('context', '').strip()]))
print(f"Found {len(unique_contexts)} unique contexts to build the Milvus index.")

# Seed and sample 20 evaluations
random.seed(3407)
np.random.seed(3407)
torch.manual_seed(3407)

eval_indices = random.sample(range(len(test_data)), min(20, len(test_data)))
eval_samples = [test_data[i] for i in eval_indices]
print(f"Selected {len(eval_samples)} samples for evaluation.")


## 3. Extract Embeddings with Qwen3-Embedding-8B


In [ ]:
print("Initializing Qwen/Qwen3-Embedding-8B in float16...")
embedder = SentenceTransformer(
    'Qwen/Qwen3-Embedding-8B', 
    model_kwargs={'torch_dtype': torch.float16}, 
    trust_remote_code=True
)

print("Encoding contexts to build Milvus index...")
context_embeddings = embedder.encode(unique_contexts, show_progress_bar=True, batch_size=32)
dimension = context_embeddings.shape[1]
print(f"Embeddings extracted. Dimension: {dimension}")


## 4. Build Milvus Lite Database


In [ ]:
db_path = "/kaggle/working/milvus_rag.db"
if os.path.exists(db_path):
    os.remove(db_path)
    
print("Initializing local Milvus Lite client...")
client = MilvusClient(db_path)

collection_name = "legal_contexts"
if client.has_collection(collection_name):
    client.drop_collection(collection_name)
    
client.create_collection(
    collection_name=collection_name,
    dimension=dimension
)

print("Inserting data into Milvus...")
data_to_insert = [
    {"id": i, "vector": context_embeddings[i].tolist(), "text": unique_contexts[i]} 
    for i in range(len(unique_contexts))
]

# Insert in batches to prevent payload size limits
batch_size = 500
for i in range(0, len(data_to_insert), batch_size):
    client.insert(collection_name=collection_name, data=data_to_insert[i:i+batch_size])
    
print(f"Milvus database built successfully with {len(data_to_insert)} records.")


## 5. Retrieve Contexts and Save to JSON


In [ ]:
def retrieve_contexts(query, k=5):
    query_emb = embedder.encode([query]).tolist()[0]
    res = client.search(
        collection_name=collection_name,
        data=[query_emb],
        limit=k,
        output_fields=["text"]
    )
    
    # Extract the 'text' field from the search results
    retrieved = [hit['entity']['text'] for hit in res[0]]
    return "\n\n".join(retrieved)

rag_eval_data = []
for sample in eval_samples:
    instruction = sample['instruction']
    retrieved_context = retrieve_contexts(instruction, k=5)
    rag_eval_data.append({
        'instruction': instruction,
        'original_context': sample.get('context', ''),
        'rag_context': retrieved_context,
        'response': sample['response']
    })

output_path = "/kaggle/working/rag_eval_data.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(rag_eval_data, f, indent=4, ensure_ascii=False)

print(f"Successfully pre-computed and saved {len(rag_eval_data)} Milvus RAG samples to {output_path}")
print("Notebook 1 execution complete! You can now run Notebook 2.")
